In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy

## Inspect nwb file

In [3]:
from pynwb import NWBHDF5IO

import sys
sys.path.append('..')

In [4]:
import warnings

nwb_path = r"X:\Personnel\MaryBeth\OpenScope\001568\sub-817335\sub-817335_ses-ecephys-817335-2025-08-27-14-46-51_ecephys.nwb"

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Ignoring cached namespace")
    io = NWBHDF5IO(nwb_path, "r", load_namespaces=True)
    nwb = io.read()

print(nwb)

root pynwb.file.NWBFile at 0x1829205289152
Fields:
  acquisition: {
    raw_running_wheel_rotation <class 'pynwb.base.TimeSeries'>,
    running_wheel_signal_voltage <class 'pynwb.base.TimeSeries'>,
    running_wheel_supply_voltage <class 'pynwb.base.TimeSeries'>
  }
  devices: {
    Device <class 'pynwb.device.Device'>,
    ProbeA <class 'pynwb.device.Device'>,
    ProbeB <class 'pynwb.device.Device'>,
    ProbeC <class 'pynwb.device.Device'>,
    ProbeE <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    ProbeA <class 'pynwb.ecephys.ElectrodeGroup'>,
    ProbeB <class 'pynwb.ecephys.ElectrodeGroup'>,
    ProbeC <class 'pynwb.ecephys.ElectrodeGroup'>,
    ProbeE <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'hdmf.common.table.DynamicTable'>
  file_create_date: [datetime.datetime(2025, 9, 22, 21, 38, 43, 68254, tzinfo=tzutc())]
  identifier: a5c81890-1b04-4c80-903a-dc9f5137970c
  institution: Allen Institute for Neural Dynamics
  intervals: {
    dri

In [5]:
print("\n--- acquisition ---")
print(list(nwb.acquisition.keys()))

print("\n--- processing ---")
print(list(nwb.processing.keys()))

print("\n--- units columns ---")
print(nwb.units.colnames)

print("\n--- electrodes columns ---")
print(nwb.electrodes.colnames)

print("\n--- intervals ---")
print(list(nwb.intervals.keys()))


--- acquisition ---
['raw_running_wheel_rotation', 'running_wheel_signal_voltage', 'running_wheel_supply_voltage']

--- processing ---
['ecephys', 'eye_tracking', 'running', 'stimulus']

--- units columns ---
('spike_times', 'electrodes', 'waveform_mean', 'waveform_sd', 'unit_name', 'drift_ptp', 'decoder_label', 'nn_hit_rate', 'snr', 'depth', 'repolarization_slope', 'firing_rate', 'rp_contamination', 'd_prime', 'amplitude_cv_median', 'amplitude_cutoff', 'shank', 'l_ratio', 'drift_mad', 'nn_miss_rate', 'num_negative_peaks', 'amplitude', 'sliding_rp_violation', 'estimated_z', 'default_qc', 'original_cluster_id', 'drift_std', 'presence_ratio', 'peak_to_valley', 'extremum_channel_index', 'num_spikes', 'velocity_above', 'decoder_probability', 'recovery_slope', 'peak_trough_ratio', 'firing_range', 'isolation_distance', 'amplitude_median', 'half_width', 'isi_violations_ratio', 'silhouette', 'sync_spike_8', 'exp_decay', 'ks_unit_id', 'spread', 'amplitude_cv_range', 'estimated_y', 'estimated_x

In [12]:
rf_stim_table = nwb.intervals["drifting_gratings_field_block_presentations"].to_dataframe()
print(rf_stim_table)

# look at the unique stimulus conditions
print()
print("Unique orientations:", rf_stim_table.orientation.unique())
print("Unique spatial frequencies:", rf_stim_table.spatial_frequency.unique())
print("Unique temporal frequencies:", rf_stim_table.temporal_frequency.unique())

# total unique combos of the three parameters
unique_combos = rf_stim_table[["orientation", "spatial_frequency", "temporal_frequency"]].drop_duplicates()
print("Total number of unique stimulus conditions:", len(unique_combos))
print(unique_combos)

       start_time    stop_time                      stim_name    stim_type  \
id                                                                           
0     1268.840779  1269.841631  drifting_gratings_field_block  GratingStim   
1     1271.092670  1272.093511  drifting_gratings_field_block  GratingStim   
2     1273.344560  1274.345400  drifting_gratings_field_block  GratingStim   
3     1275.596451  1276.597290  drifting_gratings_field_block  GratingStim   
4     1277.848331  1278.849169  drifting_gratings_field_block  GratingStim   
...           ...          ...                            ...          ...   
1495  4635.544247  4636.545076  drifting_gratings_field_block  GratingStim   
1496  4637.796129  4638.796971  drifting_gratings_field_block  GratingStim   
1497  4640.048005  4641.048858  drifting_gratings_field_block  GratingStim   
1498  4642.299900  4643.300740  drifting_gratings_field_block  GratingStim   
1499  4644.551791  4645.552630  drifting_gratings_field_block  G

## Write function to get ephys statistics

In [20]:
def presentationwise_spike_times(nwb, stim_table, stimulus_presentation_ids=None, unit_ids=None):
    """
    Produce a table associating spike times with units and stimulus presentations.
    
    Based on AllenSDK implementation.
    
    Parameters
    ----------
    nwb : NWBFile
        NWB file object
    stim_table : DataFrame
        Stimulus presentation table
    stimulus_presentation_ids : array-like, optional
        Filter to these stimulus presentations
    unit_ids : array-like, optional
        Filter to these units
    
    Returns
    -------
    pandas.DataFrame :
        Index: spike_time (float)
        Columns: stimulus_presentation_id, unit_id, time_since_stimulus_presentation_onset
    """
    if stimulus_presentation_ids is not None:
        stim_table = stim_table.loc[stimulus_presentation_ids]
    
    # Get units
    units_table = nwb.units.to_dataframe()
    if unit_ids is None:
        unit_ids = units_table.index.values
    
    # Create presentation_times array (alternating start/stop times)
    presentation_times = np.zeros([stim_table.shape[0] * 2])
    presentation_times[::2] = np.array(stim_table['start_time'])
    presentation_times[1::2] = np.array(stim_table['stop_time'])
    
    all_presentation_ids = np.array(stim_table.index.values)
    
    presentation_ids = []
    unit_ids_list = []
    spike_times_list = []
    
    for unit_id in unit_ids:
        # Retrieve spike times for this unit
        unit_row_index = units_table.index.get_loc(unit_id)
        data = nwb.units['spike_times'][unit_row_index]
        
        # Find which presentation each spike belongs to using searchsorted
        indices = np.searchsorted(presentation_times, data) - 1
        
        index_valid = indices % 2 == 0
        
        # Get presentation IDs
        presentations = all_presentation_ids[np.floor(indices / 2).astype(int)]
        
        # Sort by presentation
        sorder = np.argsort(presentations)
        presentations = presentations[sorder]
        index_valid = index_valid[sorder]
        data = data[sorder]
        
        # Find boundaries between different presentations
        changes = np.where(np.ediff1d(presentations, to_begin=1, to_end=1))[0]
        
        for ii, jj in zip(changes[:-1], changes[1:]):
            values = data[ii:jj][index_valid[ii:jj]]
            if values.size == 0:
                continue
            unit_ids_list.append(np.zeros([values.size]) + unit_id)
            presentation_ids.append(np.zeros([values.size]) + presentations[ii])
            spike_times_list.append(values)
    
    if not spike_times_list:
        # If there are no spikes, return empty DataFrame
        return pd.DataFrame(columns=[
            'stimulus_presentation_id',
            'unit_id',
            'time_since_stimulus_presentation_onset'])
    
    pres_ids = np.concatenate(presentation_ids).astype(int)
    spike_df = pd.DataFrame({
        'stimulus_presentation_id': pres_ids,
        'unit_id': np.concatenate(unit_ids_list).astype(int)
    }, index=pd.Index(np.concatenate(spike_times_list), name='spike_time'))
    
    # Add time since stimulus presentation onset
    onset_times = stim_table.loc[all_presentation_ids, "start_time"]
    spikes_with_onset = spike_df.join(onset_times, on=["stimulus_presentation_id"])
    spikes_with_onset["time_since_stimulus_presentation_onset"] = (
        spikes_with_onset.index - spikes_with_onset["start_time"]
    )
    spikes_with_onset.sort_values(by='spike_time', axis=0, inplace=True)
    spikes_with_onset.drop(columns=["start_time"], inplace=True)
    
    return spikes_with_onset

# from lines 1491-1500 AllenSDK/allensdk/brain_observatory/ecephys/ecephys_session.py
# https://github.com/AllenInstitute/AllenSDK/blob/a9b5c685396126d9748f1ccecf7c00f440569f69/allensdk/brain_observatory/ecephys/ecephys_session.py
def _extract_summary_count_statistics(index, group):
    """
    Extract summary statistics for spike counts.
    
    Based on AllenSDK implementation.
    """
    return {
        "stimulus_condition_id": index[0],
        "unit_id": index[1],
        "spike_count": group["spike_count"].sum(),
        "stimulus_presentation_count": group.shape[0],
        "spike_mean": np.mean(group["spike_count"].values),
        "spike_std": np.std(group["spike_count"].values, ddof=1),
        "spike_sem": scipy.stats.sem(group["spike_count"].values)
    }


# from lines 891-980 AllenSDK/allensdk/brain_observatory/ecephys/ecephys_session.py
# https://github.com/AllenInstitute/AllenSDK/blob/a9b5c685396126d9748f1ccecf7c00f440569f69/allensdk/brain_observatory/ecephys/ecephys_session.py
def conditionwise_spike_statistics(nwb, stimulus_block='drifting_gratings_field_block_presentations',
                                   stimulus_presentation_ids=None, unit_ids=None):
    """
    Calculate spike statistics grouped by stimulus condition.
    
    Based on AllenSDK implementation.
    
    Parameters
    ----------
    nwb : NWBFile
        NWB file object
    stimulus_block : str
        Which stimulus block to analyze
    stimulus_presentation_ids : array-like, optional
        Filter to these stimulus presentations
    unit_ids : array-like, optional
        Filter to these units
    
    Returns
    -------
    tuple : (result_df, stimulus_conditions)
        result_df: DataFrame with spike statistics indexed by [unit_id, stimulus_condition_id]
        stimulus_conditions: DataFrame with stimulus parameters indexed by stimulus_condition_id
    """
    
    stim_table = nwb.intervals[stimulus_block].to_dataframe()
    condition_params = ['orientation', 'temporal_frequency', 'spatial_frequency', 'contrast']
    stim_table_clean = stim_table[condition_params + ['start_time', 'stop_time']].dropna(
        subset=condition_params
        ).copy()
   
    for param in condition_params:
            stim_table_clean[param] = pd.to_numeric(stim_table_clean[param], errors='coerce')
    
    stimulus_presentation_ids = stim_table_clean.index.values
    
    # Create stimulus_condition_id
    stim_table_clean['stimulus_condition_id'] = stim_table_clean.groupby(condition_params).ngroup()
    
    presentations = stim_table_clean.loc[stimulus_presentation_ids].copy()
    
    # Get presentationwise spike times
    spikes = presentationwise_spike_times(nwb, stim_table_clean, stimulus_presentation_ids, unit_ids)
    
    if spikes.empty:
        # No spikes case
        units_table = nwb.units.to_dataframe()
        if unit_ids is None:
            unit_ids = units_table.index.values
        
        spike_counts = pd.DataFrame(
            {'spike_count': 0},
            index=pd.MultiIndex.from_product([
                stimulus_presentation_ids,
                unit_ids],
                names=['stimulus_presentation_id', 'unit_id']))
    else:
        # Count spikes per presentation and unit
        spike_counts = spikes[['stimulus_presentation_id', 'unit_id']].copy()
        spike_counts["spike_count"] = np.zeros(spike_counts.shape[0])
        spike_counts = spike_counts.groupby(["stimulus_presentation_id", "unit_id"]).count()
        
        unit_ids = unit_ids if unit_ids is not None else spikes['unit_id'].unique()
        
        spike_counts = spike_counts.reindex(
            pd.MultiIndex.from_product(
                [stimulus_presentation_ids, unit_ids],
                names=['stimulus_presentation_id', 'unit_id']),
            fill_value=0)
    
    # Merge with presentations to get stimulus_condition_id
    sp = pd.merge(
        spike_counts,
        presentations[['stimulus_condition_id']],
        left_on="stimulus_presentation_id",
        right_index=True,
        how="left"
    )
    sp.reset_index(inplace=True)
    
    # Extract summary statistics
    summary = []
    for ind, gr in sp.groupby(["stimulus_condition_id", "unit_id"]):
        summary.append(_extract_summary_count_statistics(ind, gr))
    
    result_df = pd.DataFrame(summary).set_index(keys=["unit_id", "stimulus_condition_id"])
    
    # Create stimulus_conditions table
    stimulus_conditions = stim_table_clean[condition_params + ['stimulus_condition_id']].drop_duplicates(
        subset='stimulus_condition_id'
    ).set_index('stimulus_condition_id').sort_index()
    
    return result_df, stimulus_conditions

In [ ]:
# Quick test on the already-loaded single mouse
conditionwise_stats, stimulus_conditions = conditionwise_spike_statistics(
    nwb, stimulus_block='drifting_gratings_field_block_presentations'
)
print(f"conditionwise_stats shape: {conditionwise_stats.shape}")
print(conditionwise_stats.head(10))
print()
print(stimulus_conditions.head())

## Build response matrix

In [21]:
def build_response_matrix(conditionwise_stats_all, stimulus_conditions_all, unit_metadata_all):
    
    unit_ids      = pd.Index([])
    condition_ids = pd.Index([])

    for conditionwise_stats, stimulus_conditions in zip(conditionwise_stats_all, stimulus_conditions_all):
        unit_ids      = unit_ids.union(conditionwise_stats.index.get_level_values('unit_id').unique())
        condition_ids = condition_ids.union(stimulus_conditions.index)

    n_units      = len(unit_ids)
    n_conditions = len(condition_ids)
    print(f"Building matrix: {n_units} units x {n_conditions} conditions")

    response_matrix = np.zeros((n_units, n_conditions))
    
    merged_meta = {}
    for meta_dict in unit_metadata_all:
        merged_meta.update(meta_dict)

    for i, unit in enumerate(unit_ids):
        unit_data = None
        for conditionwise_stats in conditionwise_stats_all:
            try:
                unit_data = conditionwise_stats.loc[unit]['spike_mean']
                break
            except KeyError:
                continue
        if unit_data is None:
            continue
        for j, cond_idx in enumerate(condition_ids):
            try:
                response_matrix[i, j] = float(unit_data.loc[cond_idx])
            except (KeyError, TypeError):
                response_matrix[i, j] = 0.0

    return response_matrix, unit_ids, condition_ids, merged_meta

## Build response matrix for all mice

In [22]:
from pathlib import Path

data_dir   = Path(r"X:\Personnel\MaryBeth\OpenScope\001568")
output_dir = Path(r"C:\Users\MaryBeth\projects\SarvestaniLab\OpenScopeMouseV1\jupyternotebooks\ephys\results")
output_dir.mkdir(exist_ok=True)

mouse_dirs = [d for d in data_dir.iterdir() if d.is_dir() and d.name.startswith('sub-')]
print(f"Found {len(mouse_dirs)} mouse directories")

Found 8 mouse directories


In [23]:
conditionwise_stats_all = []
stimulus_conditions_all = []
unit_metadata_all       = []

for mouse_idx, mouse_dir in enumerate(mouse_dirs, 1):
    mouse_name = mouse_dir.name
    print("=" * 80)
    print(f"Processing Mouse {mouse_idx}/{len(mouse_dirs)}: {mouse_name}")
    print("=" * 80)

    io = None
    try:
        nwb_files = list(mouse_dir.glob("*.nwb"))
        if not nwb_files:
            print(f"  No NWB files found in {mouse_dir}, skipping.")
            continue
        nwb_path = nwb_files[0]

        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="Ignoring cached namespace")
            io = NWBHDF5IO(str(nwb_path), "r", load_namespaces=True)
            nwb_mouse = io.read()

        units_table = nwb_mouse.units.to_dataframe()
        print(f"  Units: {len(units_table)}")

        # Compute conditionwise spike statistics
        print(f"  Computing conditionwise spike statistics...")
        conditionwise_stats, stimulus_conditions = conditionwise_spike_statistics(
            nwb_mouse, stimulus_block='drifting_gratings_field_block_presentations',
            unit_ids=units_table.index.values
        )
        print(f"  Done: {conditionwise_stats.shape[0]} (unit, condition) entries")

        # Namespace unit IDs: {mouse}__{probe}__unit{id}
        def make_uid(uid):
            probe = str(units_table.loc[uid, 'device_name'])
            return f"{mouse_name}__{probe}__unit{uid}"

        conditionwise_stats = conditionwise_stats.copy()
        conditionwise_stats.index = conditionwise_stats.index.set_levels(
            [make_uid(uid) for uid in conditionwise_stats.index.get_level_values('unit_id').unique()],
            level='unit_id'
        )

        unit_meta = {
            make_uid(uid): {
                'mouse_name': mouse_name,
                'raw_unit_id': uid,
                'probe': str(units_table.loc[uid, 'device_name'])
            }
            for uid in units_table.index
        }

        conditionwise_stats_all.append(conditionwise_stats)
        stimulus_conditions_all.append(stimulus_conditions)
        unit_metadata_all.append(unit_meta)

    except Exception as e:
        import traceback
        print(f"  ERROR on {mouse_name}: {e}")
        traceback.print_exc()

    finally:
        if io is not None:
            io.close()
            print(f"  NWB file closed for {mouse_name}")

# Build unit metadata DataFrame
unit_metadata_df = pd.DataFrame.from_dict(
    {uid: vals for d in unit_metadata_all for uid, vals in d.items()}, orient='index'
)
unit_metadata_df.index.name = 'unit_id'

# Save conditionwise stats with raw_unit_id, mouse_name, probe joined in
conditionwise_all = pd.concat(conditionwise_stats_all)
conditionwise_all = conditionwise_all.reset_index().join(
    unit_metadata_df[['mouse_name', 'raw_unit_id', 'probe']], on='unit_id'
).set_index(['unit_id', 'stimulus_condition_id'])
conditionwise_all.to_csv(output_dir / "ephys_conditionwise_stats.csv")

# Save stimulus conditions
stimulus_conditions_all[0].to_csv(output_dir / "ephys_stimulus_conditions.csv")

# Save unit metadata
unit_metadata_df.to_csv(output_dir / "ephys_unit_metadata.csv")

# Build and save response matrix
response_matrix, unit_ids, condition_ids, merged_meta = build_response_matrix(
    conditionwise_stats_all, stimulus_conditions_all, unit_metadata_all
)

np.save(output_dir / "ephys_response_matrix.npy", response_matrix)
pd.Series(unit_ids).to_csv(output_dir / "ephys_unit_ids.csv", index=False)
pd.Series(condition_ids).to_csv(output_dir / "ephys_condition_ids.csv", index=False)

print(f"\nDone.")
print(f"  Response matrix: {response_matrix.shape}")
print(f"  Units: {len(unit_ids)}")
print(f"  Conditions: {len(condition_ids)}")

Processing Mouse 1/8: sub-810531
  Units: 2251
  Computing conditionwise spike statistics...
  Done: 225100 (unit, condition) entries
  NWB file closed for sub-810531
Processing Mouse 2/8: sub-810532
  Units: 2202
  Computing conditionwise spike statistics...
  Done: 220200 (unit, condition) entries
  NWB file closed for sub-810532
Processing Mouse 3/8: sub-813810
  Units: 2996
  Computing conditionwise spike statistics...
  Done: 299600 (unit, condition) entries
  NWB file closed for sub-813810
Processing Mouse 4/8: sub-815152
  Units: 2353
  Computing conditionwise spike statistics...
  Done: 235300 (unit, condition) entries
  NWB file closed for sub-815152
Processing Mouse 5/8: sub-816305
  Units: 2732
  Computing conditionwise spike statistics...
  Done: 273200 (unit, condition) entries
  NWB file closed for sub-816305
Processing Mouse 6/8: sub-816308
  Units: 2512
  Computing conditionwise spike statistics...
  Done: 251200 (unit, condition) entries
  NWB file closed for sub-81630